In [4]:
import xarray as xr
import numpy as np

topo_vars = ["elevation", "slope", "aspect_sin", "aspect_cos"]

ds_topo = xr.open_dataset("../.data/ETOPO2/topography_features_on_gridmet_masked.nc")

# land mask from dynamic data or from topo file validity
ds_train = xr.open_dataset("../.data/downscaling_splits/train.nc")
mask = ds_train["valid_mask"] == 1

ds_topo_norm = ds_topo.copy()
topo_stats = {}

# normalize only these
for v in ["elevation", "slope"]:
    arr = ds_topo[v].where(mask)
    mean = float(arr.mean().values)
    std = float(arr.std().values)

    topo_stats[v] = {"mean": mean, "std": std}
    ds_topo_norm[v] = ((ds_topo[v] - mean) / std).where(mask, 0.0).fillna(0.0).astype("float32")

# keep aspect as-is, just fill invalid with 0
for v in ["aspect_sin", "aspect_cos"]:
    ds_topo_norm[v] = ds_topo[v].where(mask, 0.0).fillna(0.0).astype("float32")

In [ ]:
topo_norm_path = "../.data/ETOPO2/topography_features_on_gridmet_masked_norm.nc"
ds_topo_norm[topo_vars].to_netcdf(topo_norm_path)
print(f"Saved {topo_norm_path}")

Saved ../.data/ETOPO2/topography_features_on_gridmet_masked_norm.nc


In [7]:
topo_stats_ds = xr.Dataset(
    {
        "elevation_mean": xr.DataArray(topo_stats["elevation"]["mean"]),
        "elevation_std": xr.DataArray(topo_stats["elevation"]["std"]),
        "slope_mean": xr.DataArray(topo_stats["slope"]["mean"]),
        "slope_std": xr.DataArray(topo_stats["slope"]["std"]),
    }
)

topo_stats_ds.to_netcdf("../.data/ETOPO2/topography_feature_stats.nc")